**Task
: News Topic Classifier Using BERT**

---

**Objective:**

Fine-tune a transformer model (e.g., BERT) to classify news headlines into topic categories.
Dataset:
AG News Dataset (Available on Hugging Face Datasets)

**Instructions:**

● Tokenize and preprocess the dataset

● Fine-tune the bert-base-uncased model using Hugging Face Transformers

● Evaluate the model using accuracy and F1-score

● Deploy the model using Streamlit or Gradio for live interaction

---

---


In [ ]:
!pip install transformers datasets evaluate accelerate scikit-learn gradio torch

In [ ]:
import numpy as np
import pandas as pd
import torch
import evaluate
import gradio as gr
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

# ==========================================
# 1. DATASET LOADING & SUBSET SELECTION
# ==========================================

# Download the AG News dataset from Hugging Face
datasets = load_dataset("ag_news")

# Shuffle and slice a smaller subset to optimize training time and compute resources
small_train_data = datasets["train"].shuffle(seed=42).select(range(5000))
small_test_data = datasets["test"].shuffle(seed=42).select(range(1000))

# ==========================================
# 2. TOKENIZATION & PREPROCESSING
# ==========================================

# Load the pre-trained BERT tokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# Define mapping function to truncate/pad headlines to a consistent length
def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=64)

# Apply tokenization across the subsets in batches
tokenized_train = small_train_data.map(tokenize_function, batched=True)
tokenized_test = small_test_data.map(tokenize_function, batched=True)

# ==========================================
# 3. MODEL CONFIGURATION & TRAINING ARGS
# ==========================================

# Initialize BERT sequence classifier with 4 target categories
model = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=4)

# Configure model training hyperparameters
training_args = TrainingArguments(
    output_dir="./results",          # Directory where execution checkpoints are saved
    eval_strategy="epoch",           # Evaluate performance metrics at the end of each epoch
    learning_rate=2e-5,              # Initial learning rate optimized for BERT fine-tuning
    per_device_train_batch_size=16,  # Batch size for training execution
    per_device_eval_batch_size=16,   # Batch size for evaluation validation
    num_train_epochs=2,              # Total number of training passes over the dataset
    weight_decay=0.01,               # Strength of weight decay regularization to avoid overfitting
)

# ==========================================
# 4. EVALUATION METRICS DEFINITION
# ==========================================

# Load standard accuracy and F1 metrics tracking frameworks
metric_acc = evaluate.load("accuracy")
metric_f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    # Calculate performance metrics
    acc = metric_acc.compute(predictions=predictions, references=labels)["accuracy"]
    f1 = metric_f1.compute(predictions=predictions, references=labels, average="weighted")["f1"]

    return {"accuracy": acc, "f1": f1}

# ==========================================
# 5. MODEL TRAINING & SAVING FLOW
# ==========================================

# Package the configured architecture components inside the Trainer framework
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    compute_metrics=compute_metrics,
)

# Execute the fine-tuning loop
print("Starting training process...")
trainer.train()

# Persist the fine-tuned model weights and matching tokenization configurations locally
print("Saving fine-tuned model artifacts...")
model.save_pretrained("./my_bert_model")
tokenizer.save_pretrained("./my_bert_model")
print("Model artifacts successfully saved!")

# ==========================================
# 6. APP INFERENCE LOGIC (GRADIO WEB DEPLOYMENT)
# ==========================================

# Load the locally stored fine-tuned model and tokenizer artifacts for inference
model_path = "./my_bert_model"
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(model_path)

# Define explicit descriptive targets based on the AG News class index map
labels = ["World", "Sports", "Business", "Sci/Tech"]

def classify_news(headline):
    # Convert string text sequence to tensor inputs wrapped for PyTorch execution
    inputs = tokenizer(headline, return_tensors="pt", truncation=True, padding=True, max_length=64)

    # Evaluate input tokens without building gradients to ensure faster pipeline calculation
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits
        # Convert logits array into soft probability scores ranging from 0 to 1
        probabilities = torch.nn.functional.softmax(logits, dim=-1)[0]

    # Return dictionary structured with readable tags matched with confidence distribution scores
    return {labels[i]: float(probabilities[i]) for i in range(4)}

# Configure the front-end layout components for the interactive interface
demo = gr.Interface(
    fn=classify_news,
    inputs=gr.Textbox(lines=2, placeholder="Type an English news headline here...", label="News Headline"),
    outputs=gr.Label(num_top_classes=4, label="Predicted Category"),
    title="📰 News Topic Classifier (BERT)",
    description="Fine-tuned BERT model classifying news headlines into World, Sports, Business, or Sci/Tech classes."
)

# Main runtime block handler to initialize web server engine
if __name__ == "__main__":
    demo.launch()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/18.6M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/1.23M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

...

Saving fine-tuned model artifacts...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model artifacts successfully saved!


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://875f889a0a1b16739e.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
